# Phase 2: Chunking & Embedding All Documents

This notebook:
1. Loads all Docling JSON documents
2. Chunks them with the strategy selected by `CHUNKING`
3. Embeds with BGE-large
4. Stores in Milvus vector database
5. Tests retrieval queries
6. Generates answers using Gemini

| `CHUNKING` | Chunker | Milvus DB / collection | Gemini model |
|---|---|---|---|
| `"hierarchical"` | HierarchicalChunker for every document | `milvus_all_docs.db` / `rag_all_docs` | `gemini-2.5-flash-lite` |
| `"hybrid"` | HybridChunker with token-aware refinements | `milvus_all_docs_hybrid.db` / `rag_all_docs_hybrid` | `gemini-2.5-flash-lite` |
| `"mixed"` | HybridChunker for PDF, DOCX, images; HierarchicalChunker for XLSX files | `milvus_all_docs_mixed.db` / `rag_all_docs_mixed` | `gemini-2.5-flash` |

In [ ]:
# Chunking strategy: "hierarchical", "hybrid" or "mixed" (see the table above)
CHUNKING = "hierarchical"

In [ ]:
CHUNKING_SETTINGS = {
    "hierarchical": {"suffix": "",        "gemini_model": "gemini-2.5-flash-lite", "financial_prompt": False, "insert_batch_size": None},
    "hybrid":       {"suffix": "_hybrid", "gemini_model": "gemini-2.5-flash-lite", "financial_prompt": True,  "insert_batch_size": 1000},
    "mixed":        {"suffix": "_mixed",  "gemini_model": "gemini-2.5-flash",      "financial_prompt": True,  "insert_batch_size": None},
}
if CHUNKING not in CHUNKING_SETTINGS:
    raise ValueError(f"Unknown CHUNKING {CHUNKING!r}; choose one of {sorted(CHUNKING_SETTINGS)}")
chunking_settings = CHUNKING_SETTINGS[CHUNKING]

SUFFIX = chunking_settings["suffix"]                          # appended to the Milvus DB, collection and report names
GEMINI_MODEL = chunking_settings["gemini_model"]
USE_FINANCIAL_PROMPT = chunking_settings["financial_prompt"]  # financial-analyst prompt instead of the plain QA prompt
INSERT_BATCH_SIZE = chunking_settings["insert_batch_size"]    # None: insert all records in one call

print(f"Chunking strategy: {CHUNKING}")

## Cell 1: Imports & GPU Check

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import json
from pathlib import Path
from pymilvus import MilvusClient
from tqdm import tqdm
from docling.chunking import HybridChunker, HierarchicalChunker
from docling_core.types.doc import DoclingDocument
import time
from datetime import datetime
import google.generativeai as genai
import os

# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not available, using CPU")

## Cell 2: Load BGE Embedding Model

In [ ]:
# Load BGE embedding model
model_name = "BAAI/bge-large-en-v1.5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
embedding_model = SentenceTransformer(model_name, device=device)

# Test embedding to get dimension
test_embedding = embedding_model.encode(["test"])
embedding_dim = len(test_embedding[0])

print(f"\nModel loaded successfully!")
print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Max sequence length: {embedding_model.max_seq_length}")

## Cell 3: Configuration

In [ ]:
# Configuration
INPUT_DIR = Path("docling_outputs/json")
MILVUS_DB = f"milvus_all_docs{SUFFIX}.db"
COLLECTION_NAME = f"rag_all_docs{SUFFIX}"
BATCH_SIZE = 32
EMBEDDING_DIM = 1024

# Metadata fields
METADATA_FIELDS = ["source_file", "doc_type", "chunk_id", "total_chunks"]
if CHUNKING == "mixed":
    METADATA_FIELDS.append("chunker_used")

print("Configuration:")
print(f"  Input directory: {INPUT_DIR}")
print(f"  Milvus DB: {MILVUS_DB}")
print(f"  Collection name: {COLLECTION_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Metadata fields: {METADATA_FIELDS}")

## Cell 4: Load All Docling JSON Documents

In [ ]:
# Scan for all JSON files
json_files = sorted(INPUT_DIR.glob("*.json"))

print(f"Found {len(json_files)} JSON files in {INPUT_DIR}\n")

# Load all documents
documents = {}
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as f:
        doc_data = json.load(f)
    documents[json_file.stem] = doc_data
    print(f"Loaded: {json_file.name}")

print(f"\nTotal documents loaded: {len(documents)}")
print(f"\nDocument files:")
for filename in documents.keys():
    print(f"  - {filename}")

## Cell 5: Chunk All Documents

In [ ]:
if CHUNKING == "hierarchical":
    # Initialize HierarchicalChunker
    chunker = HierarchicalChunker()

elif CHUNKING == "hybrid":
    # Initialize HybridChunker with BGE tokenizer
    # HybridChunker does tokenization-aware refinements on top of document layout chunking
    # It ensures chunks respect the tokenizer's max_tokens limit (512 for BGE-large)
    chunker = HybridChunker(
        tokenizer="BAAI/bge-large-en-v1.5",  # Match embedding model tokenizer
        merge_peers=True  # Merge undersized chunks with same metadata
    )

    print(f"HybridChunker initialized:")
    print(f"  Tokenizer: BAAI/bge-large-en-v1.5")
    print(f"  Max tokens: {chunker.max_tokens}")
    print(f"  Merge peers: True")
    print()

else:
    # Initialize both chunkers
    # HybridChunker: token-aware, respects 512 token limit (for PDF, DOCX, etc.)
    hybrid_chunker = HybridChunker(
        tokenizer="BAAI/bge-large-en-v1.5",
        merge_peers=True
    )

    # HierarchicalChunker: structure-aware, preserves table coherence (for XLSX)
    hierarchical_chunker = HierarchicalChunker()

    print(f"Chunkers initialized:")
    print(f"  HybridChunker:")
    print(f"    - Tokenizer: BAAI/bge-large-en-v1.5")
    print(f"    - Max tokens: {hybrid_chunker.max_tokens}")
    print(f"    - Merge peers: True")
    print(f"    - Used for: PDF, DOCX, PPTX, images, etc.")
    print(f"  HierarchicalChunker:")
    print(f"    - Used for: XLSX files (preserves table structure)")
    print()


all_chunks = []
doc_type_counts = {}
if CHUNKING == "mixed":
    chunker_usage_stats = {"HybridChunker": 0, "HierarchicalChunker": 0}
    print("Chunking all documents with mixed strategy...\n")
else:
    print("Chunking all documents...\n")

for filename, doc_data in tqdm(documents.items(), desc="Chunking documents"):
    # Reconstruct Docling Document from JSON
    doc = DoclingDocument.model_validate(doc_data)
    
    # Get document type from origin or filename
    doc_type = "unknown"
    if "origin" in doc_data and "filename" in doc_data["origin"]:
        original_filename = doc_data["origin"]["filename"]
        doc_type = Path(original_filename).suffix.lstrip('.').lower() or "unknown"
    else:
        # Fallback to inferring from filename
        if "pdf" in filename.lower():
            doc_type = "pdf"
        elif "docx" in filename.lower():
            doc_type = "docx"
        elif "pptx" in filename.lower():
            doc_type = "pptx"
        elif "xlsx" in filename.lower():
            doc_type = "xlsx"
        elif "html" in filename.lower():
            doc_type = "html"
        elif "md" in filename.lower():
            doc_type = "md"
    
    # Choose chunker based on document type (mixed strategy)
    if CHUNKING == "mixed":
        if doc_type == "xlsx":
            chunker = hierarchical_chunker
            chunker_name = "HierarchicalChunker"
        else:
            chunker = hybrid_chunker
            chunker_name = "HybridChunker"
    
    # Apply chunker
    chunk_iter = chunker.chunk(doc)
    doc_chunks = list(chunk_iter)
    
    # Track chunker usage
    if CHUNKING == "mixed":
        chunker_usage_stats[chunker_name] += len(doc_chunks)
    
    # Extract chunk texts and metadata
    for idx, chunk in enumerate(doc_chunks):
        chunk_text = chunk.text
        record = {
            "text": chunk_text,
            "source_file": filename,
            "doc_type": doc_type,
            "chunk_id": idx,
            "total_chunks": len(doc_chunks)
        }
        if CHUNKING == "mixed":
            record["chunker_used"] = chunker_name
        all_chunks.append(record)
    
    # Track counts by doc type
    if doc_type not in doc_type_counts:
        doc_type_counts[doc_type] = 0
    doc_type_counts[doc_type] += len(doc_chunks)

print(f"\n{'='*60}")
print(f"Total chunks created: {len(all_chunks)}")
if CHUNKING == "mixed":
    print(f"\nChunker usage:")
    for chunker_name, count in chunker_usage_stats.items():
        percentage = (count / len(all_chunks)) * 100
        print(f"  {chunker_name}: {count} chunks ({percentage:.1f}%)")
print(f"\nBreakdown by document type:")
for doc_type, count in sorted(doc_type_counts.items()):
    print(f"  {doc_type}: {count} chunks")
print(f"{'='*60}")

## Cell 6: Batch Embed All Chunks

In [ ]:
# Extract all chunk texts
all_texts = [chunk["text"] for chunk in all_chunks]

print(f"Embedding {len(all_texts)} chunks...")
start_time = time.time()

# Batch embed all texts
embeddings = embedding_model.encode(
    all_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Convert to list of lists
embeddings_list = embeddings.tolist()

end_time = time.time()
elapsed_time = end_time - start_time

print(f"\n{'='*60}")
print(f"Embedding complete!")
print(f"Total chunks embedded: {len(embeddings_list)}")
print(f"Time taken: {elapsed_time:.2f} seconds")
print(f"Average time per chunk: {elapsed_time/len(embeddings_list)*1000:.2f} ms")
print(f"Embedding shape: {embeddings.shape}")
print(f"{'='*60}")

## Cell 7: Initialize Milvus & Create Collection

In [ ]:
# Initialize Milvus client with better error handling
try:
    # Try to close any existing connection
    if 'client' in globals():
        try:
            client.close()
            print("Closed existing Milvus connection")
        except:
            pass
except:
    pass

# Initialize new client
client = MilvusClient(MILVUS_DB)

# Drop collection if exists (fresh start)
if client.has_collection(COLLECTION_NAME):
    print(f"Dropping existing collection '{COLLECTION_NAME}'...")
    client.drop_collection(COLLECTION_NAME)

# Create collection with schema
print(f"Creating collection '{COLLECTION_NAME}'...")
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBEDDING_DIM,
    metric_type="IP",  # Inner Product
    consistency_level="Strong"
)

print(f"\nCollection '{COLLECTION_NAME}' created successfully!")
print(f"  Dimension: {EMBEDDING_DIM}")
print(f"  Metric type: IP (Inner Product)")
print(f"  Consistency level: Strong")
print(f"  Metadata fields: {METADATA_FIELDS}")

## Cell 8: Insert Data into Milvus

In [ ]:
# Prepare data for insertion
data_to_insert = []

for idx, (chunk, embedding) in enumerate(zip(all_chunks, embeddings_list)):
    data_to_insert.append({
        "id": idx,
        "vector": embedding,
        "text": chunk["text"],
        "source_file": chunk["source_file"],
        "doc_type": chunk["doc_type"],
        "chunk_id": chunk["chunk_id"],
        "total_chunks": chunk["total_chunks"]
    })
    if CHUNKING == "mixed":
        data_to_insert[-1]["chunker_used"] = chunk["chunker_used"]

if INSERT_BATCH_SIZE is None:
    print(f"Inserting {len(data_to_insert)} records into Milvus...")
    start_time = time.time()

    # Insert all at once
    insert_result = client.insert(
        collection_name=COLLECTION_NAME,
        data=data_to_insert
    )

    end_time = time.time()

    print(f"\n{'='*60}")
    print(f"Insertion complete!")
    print(f"Records inserted: {insert_result['insert_count']}")
    print(f"Time taken: {end_time - start_time:.2f} seconds")
    print(f"{'='*60}")
else:
    # Insert in batches to avoid connection timeout
    print(f"Inserting {len(data_to_insert)} records into Milvus in batches...")
    print(f"Total records: {len(data_to_insert)}")

    start_time = time.time()
    total_inserted = 0

    for i in range(0, len(data_to_insert), INSERT_BATCH_SIZE):
        batch = data_to_insert[i:i + INSERT_BATCH_SIZE]
        batch_num = (i // INSERT_BATCH_SIZE) + 1
        total_batches = (len(data_to_insert) + INSERT_BATCH_SIZE - 1) // INSERT_BATCH_SIZE
    
        print(f"Inserting batch {batch_num}/{total_batches} ({len(batch)} records)...", end=" ")
    
        try:
            insert_result = client.insert(
                collection_name=COLLECTION_NAME,
                data=batch
            )
            total_inserted += insert_result['insert_count']
            print(f"✓ ({insert_result['insert_count']} inserted)")
        except Exception as e:
            print(f"✗ Error: {str(e)[:100]}")
            # Continue with next batch even if this one fails

    end_time = time.time()

    print(f"\n{'='*60}")
    print(f"Insertion complete!")
    print(f"Total records inserted: {total_inserted}/{len(data_to_insert)}")
    print(f"Time taken: {end_time - start_time:.2f} seconds")
    print(f"Average time per batch: {(end_time - start_time)/total_batches:.2f} seconds")
    print(f"{'='*60}")

## Cell 9: Test Search - Financial Statements Query

In [ ]:
# Query 1: Financial statements
query_text = "What are the financial statements prepared according to Indian Accounting Standards for FY 23-24?"

print(f"Query: {query_text}\n")
print("Embedding query...")
query_embedding = embedding_model.encode([query_text])[0].tolist()

print("Searching Milvus...")
search_results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_embedding],
    limit=10,
    output_fields=["text"] + METADATA_FIELDS
)

print(f"\n{'='*80}")
print(f"Top 10 Results:")
print(f"{'='*80}\n")

for idx, result in enumerate(search_results[0], 1):
    print(f"Result {idx}:")
    print(f"  Source: {result['entity']['source_file']}")
    print(f"  Doc Type: {result['entity']['doc_type']}")
    if CHUNKING == "mixed":
        print(f"  Chunker: {result['entity']['chunker_used']}")
    print(f"  Chunk: {result['entity']['chunk_id']+1}/{result['entity']['total_chunks']}")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Text: {result['entity']['text'][:200]}...")
    print()

## Cell 10: Test Search - Balance Sheet Query

In [ ]:
# Query 2: Balance sheet
query_text = "Show me balance sheet information for ExampleCo"

print(f"Query: {query_text}\n")
print("Embedding query...")
query_embedding = embedding_model.encode([query_text])[0].tolist()

print("Searching Milvus...")
search_results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_embedding],
    limit=10,
    output_fields=["text"] + METADATA_FIELDS
)

print(f"\n{'='*80}")
print(f"Top 10 Results:")
print(f"{'='*80}\n")

for idx, result in enumerate(search_results[0], 1):
    print(f"Result {idx}:")
    print(f"  Source: {result['entity']['source_file']}")
    print(f"  Doc Type: {result['entity']['doc_type']}")
    if CHUNKING == "mixed":
        print(f"  Chunker: {result['entity']['chunker_used']}")
    print(f"  Chunk: {result['entity']['chunk_id']+1}/{result['entity']['total_chunks']}")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Text: {result['entity']['text'][:200]}...")
    print()

## Cell 11: Collection Statistics & Summary

In [ ]:
# Get collection statistics
stats = client.get_collection_stats(COLLECTION_NAME)

# Calculate breakdown by doc_type
doc_type_breakdown = {}
for chunk in all_chunks:
    doc_type = chunk["doc_type"]
    if doc_type not in doc_type_breakdown:
        doc_type_breakdown[doc_type] = 0
    doc_type_breakdown[doc_type] += 1

# Calculate per-document breakdown
per_doc_breakdown = {}
for chunk in all_chunks:
    source = chunk["source_file"]
    if source not in per_doc_breakdown:
        per_doc_breakdown[source] = 0
    per_doc_breakdown[source] += 1

print(f"\n{'='*80}")
print(f"COLLECTION STATISTICS")
print(f"{'='*80}")
print(f"\nCollection Name: {COLLECTION_NAME}")
print(f"Total Vectors: {stats['row_count']}")
print(f"Embedding Dimension: {EMBEDDING_DIM}")

print(f"\nBreakdown by Document Type:")
for doc_type, count in sorted(doc_type_breakdown.items()):
    percentage = (count / len(all_chunks)) * 100
    print(f"  {doc_type}: {count} chunks ({percentage:.1f}%)")

print(f"\nPer-Document Chunk Counts:")
for source, count in sorted(per_doc_breakdown.items()):
    print(f"  {source}: {count} chunks")

avg_chunks = len(all_chunks) / len(documents)
print(f"\nAverage chunks per document: {avg_chunks:.1f}")
print(f"Total documents processed: {len(documents)}")
print(f"Total chunks created: {len(all_chunks)}")

# Estimate collection size
vector_size = len(all_chunks) * EMBEDDING_DIM * 4 / (1024 * 1024)  # 4 bytes per float32, convert to MB
print(f"\nEstimated collection size: {vector_size:.2f} MB (vectors only)")
print(f"{'='*80}")

## Cell 12: Save Chunking Report

In [ ]:
# Strategy-specific fields for the report
if CHUNKING == "hybrid":
    CHUNKING_REPORT_FIELDS = {
        "chunking_method": "HybridChunker",
        "chunking_tokenizer": "BAAI/bge-large-en-v1.5",
        "chunking_max_tokens": chunker.max_tokens,
    }
elif CHUNKING == "mixed":
    CHUNKING_REPORT_FIELDS = {
        "chunking_method": "Mixed (Hybrid + Hierarchical)",
        "chunking_strategy": {
            "xlsx_files": "HierarchicalChunker (structure-preserving)",
            "other_files": "HybridChunker (token-aware, max 512 tokens)"
        },
        "chunker_usage_stats": chunker_usage_stats,
    }
else:
    CHUNKING_REPORT_FIELDS = {}

# Create chunking report
report = {
    "processing_info": {
        "timestamp": datetime.now().isoformat(),
        "total_documents_processed": len(documents),
        "total_chunks_created": len(all_chunks),
        "embedding_model": model_name,
        **CHUNKING_REPORT_FIELDS,
        "milvus_collection": COLLECTION_NAME,
        "milvus_db_file": MILVUS_DB
    },
    "chunking_statistics": {
        "average_chunks_per_document": len(all_chunks) / len(documents),
        "chunks_by_doc_type": doc_type_breakdown,
        "chunks_by_document": per_doc_breakdown
    },
    "embedding_info": {
        "model_name": model_name,
        "dimension": EMBEDDING_DIM,
        "batch_size": BATCH_SIZE,
        "device": device
    },
    "milvus_info": {
        "collection_name": COLLECTION_NAME,
        "metric_type": "IP",
        "total_vectors": stats['row_count'],
        "metadata_fields": METADATA_FIELDS
    },
    "document_list": list(documents.keys())
}

# Save report
report_path = Path(f"docling_outputs/chunking_report{SUFFIX}.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Chunking report saved to: {report_path}")
print(f"\nReport summary:")
print(json.dumps(report, indent=2))

## Cell 13: Initialize Gemini for Answer Generation

In [ ]:
# Configure Gemini API
# Note: Set your GEMINI_API_KEY or GOOGLE_API_KEY environment variable
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not found in environment variables")
    print("Please set GEMINI_API_KEY or GOOGLE_API_KEY to use Gemini for answer generation")
    print("Example: export GEMINI_API_KEY='your-api-key-here'")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel(GEMINI_MODEL)
    
    print("Gemini initialized successfully!")
    print(f"Model: {GEMINI_MODEL}")
    print(f"Ready for RAG answer generation")

## Cell 14: RAG Function - Retrieve Context & Generate Answer

In [ ]:
def rag_query(query_text, top_k=5, show_full_chunks=True):
    """
    Complete RAG pipeline: Retrieve relevant chunks and generate answer using Gemini.
    
    Args:
        query_text: The question to answer
        top_k: Number of chunks to retrieve
        show_full_chunks: Whether to print full chunk text
    
    Returns:
        dict with answer, sources, and metadata
    """
    print(f"\n{'='*80}")
    print(f"QUERY: {query_text}")
    print(f"{'='*80}\n")
    
    # Step 1: Embed query
    print(f"Step 1: Embedding query...")
    query_embedding = embedding_model.encode([query_text])[0].tolist()
    
    # Step 2: Search vector database
    print(f"Step 2: Searching for top {top_k} relevant chunks...")
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text"] + METADATA_FIELDS
    )
    
    # Extract results
    context_texts = []
    sources = []
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNKS:")
    print(f"{'='*80}\n")
    
    for idx, result in enumerate(search_results[0], 1):
        entity = result['entity']
        chunk_text = entity['text']
        context_texts.append(chunk_text)
        
        source_info = {
            'rank': idx,
            'source_file': entity['source_file'],
            'doc_type': entity['doc_type'],
            'chunk_id': entity['chunk_id'],
            'total_chunks': entity['total_chunks'],
            'distance': result['distance'],
            'text': chunk_text
        }
        if CHUNKING == "mixed":
            source_info['chunker_used'] = entity['chunker_used']
        sources.append(source_info)
        
        # Display chunk info
        print(f"Chunk {idx}:")
        print(f"  Source: {entity['source_file']} ({entity['doc_type']})")
        if CHUNKING == "mixed":
            print(f"  Chunker: {entity['chunker_used']}")
        print(f"  Chunk: {entity['chunk_id']+1}/{entity['total_chunks']}")
        print(f"  Distance: {result['distance']:.4f}")
        if show_full_chunks:
            print(f"  Text:\n{chunk_text}")
        else:
            print(f"  Text: {chunk_text[:200]}...")
        print(f"\n{'-'*80}\n")
    
    # Step 3: Build context for Gemini
    print(f"Step 3: Building context for LLM...")
    context = "\n\n".join([
        f"[Source {i+1}: {sources[i]['source_file']}]\n{text}" 
        for i, text in enumerate(context_texts)
    ])
    
    # Step 4: Generate answer with Gemini
    print(f"Step 4: Generating answer with Gemini 2.5 Flash...\n")
    
    if USE_FINANCIAL_PROMPT:
        prompt = f"""You are an expert financial analyst AI assistant with access to retrieved document context from a RAG (Retrieval-Augmented Generation) system.

    IMPORTANT CONTEXT UNDERSTANDING:
    The context provided below consists of relevant text chunks retrieved from financial documents (balance sheets, audit reports, financial statements, XBRL filings). These chunks may contain:
    - Complete information directly answering the question
    - Partial information requiring synthesis across multiple chunks
    - Numerical data from tables requiring calculations
    - Related information from which the answer can be derived through reasoning

    YOUR CAPABILITIES:
    1. **Multi-step reasoning**: You can derive answers through logical inference even if not explicitly stated
    2. **Calculations**: You can perform arithmetic operations (addition, subtraction, multiplication, division, percentages, ratios, year-over-year changes)
    3. **Cross-chunk synthesis**: Combine information from multiple retrieved chunks to form complete answers
    4. **Financial domain knowledge**: Apply standard accounting principles and financial analysis techniques
    5. **Pattern recognition**: Identify trends, anomalies, and relationships in financial data

    RETRIEVED CONTEXT:
    {context}

    USER QUESTION: {query_text}

    INSTRUCTIONS:
    1. **Direct Information**: If the answer is explicitly stated in the context, provide it directly with source citations
    2. **Derivable Information**: If the answer requires calculation or inference:
    - Clearly explain your reasoning steps
    - Show calculations with numbers from the context
    - Cite which sources provided the input data
    3. **Incomplete Information**: If context provides partial information:
    - State what information is available
    - Explain what additional data would be needed
    - Provide the best possible answer with appropriate caveats
    4. **No Information**: If context doesn't contain relevant information:
    - Explicitly state that the information is not available in the retrieved documents
    - Do NOT make up information or use knowledge outside the provided context

    ANSWER FORMAT:
    - Begin with a clear, concise answer
    - Show calculations if performed (e.g., "Total Assets = 1,234,567 + 567,890 = 1,802,457")
    - Cite sources: [Source 1], [Source 2], etc.
    - If derived through reasoning, explain the logic
    - Use proper financial terminology

    Your response:"""
    else:
        prompt = f"""You are a helpful AI assistant that answers questions based on provided document context.

Context from documents:
{context}

Question: {query_text}

Please provide a comprehensive answer based on the context above. 
- If the context contains relevant information, synthesize it into a clear answer.
- If the context doesn't contain enough information, explicitly state that.
- Cite which sources you're using when answering."""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'question': query_text,
        'answer': answer,
        'sources': sources,
        'num_chunks_retrieved': len(context_texts)
    }

print("RAG function ready!")
print("Usage: rag_query('your question here', top_k=5, show_full_chunks=True)")

## Cell 15: Test RAG - Financial Statements Query

In [ ]:
# Test RAG with financial statements query
result1 = rag_query(
    "What are the financial statements prepared according to Indian Accounting Standards for FY 23-24?",
    top_k=5,
    show_full_chunks=True
)

## Cell 16: Test RAG - Balance Sheet Query

In [ ]:
# Test RAG with balance sheet query
result2 = rag_query(
    "Show me balance sheet information for ExampleCo",
    top_k=5,
    show_full_chunks=True
)

## Cell 17: Custom Query Interface

In [ ]:
# Use this cell to run your own custom queries
# Uncomment and modify the query below:
if CHUNKING == "hybrid":
    questions = [
                #Page 1 questions
                "Language (English/Hindi)",
                "Authorised capital of the company as on the date of filling (in Rs.)",
                "1.(a) Corporate Identity Number (CIN) of company",
                "2.(a) Name of the company",
                "2.(b) Address of the registered office of the company",
                "2.(c) e-mail ID of the company",
                "3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)",
                "4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)"
        ]
else:
    questions = [
                #Page 1 questions
                "Language (English/Hindi)"
                "Authorised capital of the company as on the date of filling (in Rs.)",
                "1.(a) Corporate Identity Number (CIN) of company",
                "2.(a) Name of the company",
                "2.(b) Address of the registered office of the company",
                "2.(c) e-mail ID of the company",
                "3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)",
                "4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)",
                "4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)",
                "5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)",
                "6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)",

                #Page 2 questions
                "7. Type of Industry",
                "8. Whether consolidated Financial State,emts are also being filed (Yes/No)",
                "9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)",
                "9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)",
                "10. Whether Secretarial Audit is applicable (Yes/No)"
                "11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)"
        ]
for question in questions:
    custom_result = rag_query(
        question,
        top_k=5,
        show_full_chunks=True
    )


In [ ]:
# Second batch of form questions (hybrid runs)
if CHUNKING == "hybrid":
    # Uncomment and modify the query below:
    questions = ["4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)",
                "5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)",
                "6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)",

                #Page 2 questions
                "7. Type of Industry",
                "8. Whether consolidated Financial State,emts are also being filed (Yes/No)",
                "9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)",
                "9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)",
                "10. Whether Secretarial Audit is applicable (Yes/No)",
                "11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)"
            ]

        

    for question in questions:
        custom_result = rag_query(
            question,
            top_k=5,
            show_full_chunks=True
        )


In [ ]:
CUSTOM_QUERY = {
    "hierarchical": "Address of the company",
    "hybrid": "the language of form filling is ___ (english/hindi)",
    "mixed": "name of the company",
}[CHUNKING]
custom_result = rag_query(
        CUSTOM_QUERY,
        top_k=5,
        show_full_chunks=True
    )

## Summary

This notebook has:
1. Loaded all Docling JSON documents from `docling_outputs/json/`
2. Chunked them with the strategy selected by `CHUNKING`:
   - `hierarchical`: HierarchicalChunker
   - `hybrid`: **HybridChunker** with token-aware refinements
   - `mixed`: **HybridChunker** for PDF, DOCX, PPTX, images (token-aware, ≤512 tokens) and **HierarchicalChunker** for XLSX files (structure-preserving)
3. Embedded all chunks using BGE-large (1024 dimensions)
4. Stored embeddings in Milvus vector database
5. Implemented semantic search across all documents
6. Integrated Gemini for answer generation
7. Created a complete RAG pipeline for question answering

### Key Features:
- **Single collection** for all document types with metadata
- **Batch embedding** for maximum performance
- **Rich metadata** including source file, doc type, chunk IDs (and chunker used for `mixed`)
- **Full RAG pipeline** with context retrieval and LLM generation
- **Reusable function** for custom queries

### Advantages of HybridChunker:
- Ensures no chunk exceeds tokenizer's max_tokens (512 for BGE)
- Merges undersized chunks intelligently
- Maintains document structure while respecting token boundaries
- Better semantic coherence for embeddings

### Mixed Chunking Strategy:
**For XLSX files (HierarchicalChunker):**
- ✅ Preserves complete table structure
- ✅ Keeps headers with data rows
- ✅ Avoids 16x fragmentation
- ⚠️ May exceed 512 tokens (BGE truncates during embedding)
- ⚠️ Acceptable because table context > completeness

**For other files (HybridChunker):**
- ✅ Respects 512 token limit
- ✅ All content gets embedded
- ✅ Merges small chunks intelligently
- ✅ Better for text-heavy documents

### Expected Results:
- XLSX chunks: ~1,608 (same as hierarchical)
- Other chunks: ~2,200 (hybrid, token-limited)
- Total: ~3,800 chunks (vs 5,086 hierarchical or 28,554 pure hybrid)

### Next Steps:
- Use the `rag_query()` function for any questions about your documents
- Experiment with different `top_k` values for retrieval
- Add metadata filtering for specific document types or chunkers
- Compare retrieval quality across the three `CHUNKING` strategies
- Export results for downstream applications